# OCR Model Selection Lab — Colab autonome

Ce notebook est une alternative complète à l’application du repo. Il ne clone pas GitHub et ne copie pas le code du projet : toute la logique benchmark est écrite directement dans les cellules.

Ordre logique :

1. Installer les dépendances.
2. Vérifier CPU/GPU.
3. Charger un dataset : exemple intégré, ZIP uploadé, ou Kaggle.
4. Télécharger/préparer les modèles.
5. Exécuter le benchmark.
6. Calculer les métriques.
7. Afficher graphiques et résultats.
8. Télécharger les exports.

Important : télécharger un modèle ne suffit pas toujours. Pour benchmarker un modèle, il faut aussi savoir l’appeler en Python. Le notebook fournit des adaptateurs génériques, mais certains modèles demandent un code spécifique.

In [ ]:
# ÉTAPE 1 — Installer les dépendances
# Exécutez cette cellule une fois. Pour PaddleOCR GPU, mettez INSTALL_PADDLE=True.

INSTALL_PADDLE = False

import subprocess, sys

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip_install(
    "pandas", "numpy", "pillow", "matplotlib", "plotly", "gradio", "tqdm",
    "python-Levenshtein", "huggingface_hub", "kagglehub", "transformers",
    "accelerate", "sentencepiece", "protobuf", "safetensors", "easyocr",
)

if INSTALL_PADDLE:
    pip_install("paddleocr", "paddlepaddle-gpu")

print("Dépendances installées.")

In [ ]:
# ÉTAPE 2 — Vérifier CPU/GPU
import os, sys, json, math, time, random, shutil, zipfile, platform
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None
    DEVICE = "cpu"

print("Python:", platform.python_version())
print("Device:", DEVICE)
if torch and DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("CPU actif. Pour GPU : Runtime > Change runtime type > T4 GPU.")

WORK_DIR = Path("/content/ocr_benchmark_standalone")
DATASET_DIR = WORK_DIR / "dataset"
UPLOAD_DIR = DATASET_DIR / "user_uploads"
RUNS_DIR = WORK_DIR / "runs"
for p in [DATASET_DIR, UPLOAD_DIR, RUNS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print("Workspace:", WORK_DIR)

## ÉTAPE 3 — Dataset

Le benchmark a besoin d’images + texte attendu exact (`ground_truth`).

Options :

- `USE_SAMPLE_DATASET=True` : mini dataset synthétique intégré au notebook.
- `IMPORT_CUSTOM_DATASET_ZIP=True` : upload d’un ZIP avec `labels.csv` ou `dataset.json`.
- `DOWNLOAD_KAGGLE_DATASET=True` : téléchargement Kaggle via `kagglehub`.

Format ZIP recommandé :

```text
mon_dataset.zip
├── labels.csv
└── images/
    ├── cheque_001.png
    └── form_001.jpg
```

Format `labels.csv` :

```csv
image_path,ground_truth,category,description
images/cheque_001.png,"texte attendu exact",bank,"chèque manuscrit"
images/form_001.jpg,"texte attendu exact",handwritten_form,"formulaire rempli"
```

In [ ]:
# ÉTAPE 3A — Configuration dataset
USE_SAMPLE_DATASET = True
IMPORT_CUSTOM_DATASET_ZIP = False
DOWNLOAD_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = ""  # exemple: "senju14/ocr-dataset-of-multi-type-documents"

ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
print("Configuration dataset prête.")

In [ ]:
# ÉTAPE 3B — Fonctions dataset

def make_sample_image(path: Path, lines, title=None, size=(1100, 760)):
    path.parent.mkdir(parents=True, exist_ok=True)
    img = Image.new("RGB", size, "#fbfaf7")
    draw = ImageDraw.Draw(img)
    try:
        font_title = ImageFont.truetype("DejaVuSans-Bold.ttf", 36)
        font = ImageFont.truetype("DejaVuSans.ttf", 28)
    except Exception:
        font_title = font = None
    y = 50
    if title:
        draw.text((50, y), title, fill="#111827", font=font_title)
        y += 70
    for line in lines:
        draw.text((50, y), line, fill="#1f2937", font=font)
        y += 44
    img.save(path)


def build_sample_dataset():
    specs = [
        ("bank", "cheque_sample.png", "BANQUE EXEMPLE\nPayez contre ce chèque\nMontant: 1 250,00 EUR\nBénéficiaire: Eric Kambire\nDate: 12/07/2026", "Chèque synthétique"),
        ("tables", "table_sample.png", "| Produit | Quantité | Prix |\n| --- | ---: | ---: |\n| Cahier | 4 | 12.00 |\n| Stylo | 10 | 7.50 |", "Table synthétique"),
        ("handwritten_form", "form_sample.png", "Nom: Eric Kambire\nAdresse: 10 rue Exemple\nTéléphone: 0600000000\nSignature: EK", "Formulaire synthétique"),
    ]
    rows = []
    for category, filename, gt, desc in specs:
        path = DATASET_DIR / "sample" / filename
        make_sample_image(path, gt.splitlines(), desc)
        rows.append({"image_path": str(path), "ground_truth": gt, "category": category, "description": desc})
    return rows


def assert_inside(child: Path, parent: Path) -> Path:
    child = child.resolve(); parent = parent.resolve()
    if child != parent and parent not in child.parents:
        raise ValueError(f"Chemin non autorisé: {child}")
    return child


def safe_extract_zip(zip_path: str, extract_dir: Path):
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = assert_inside(extract_dir / member.filename, extract_dir)
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True); continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as dest:
                shutil.copyfileobj(source, dest)


def read_records_from_folder(folder: Path):
    labels_csv = next(folder.rglob("labels.csv"), None)
    dataset_json = next(folder.rglob("dataset.json"), None)
    if labels_csv:
        records = pd.read_csv(labels_csv).fillna("").to_dict("records")
        base_dir = labels_csv.parent
    elif dataset_json:
        records = json.loads(dataset_json.read_text(encoding="utf-8"))
        base_dir = dataset_json.parent
    else:
        raise ValueError("Le dossier doit contenir labels.csv ou dataset.json")
    rows = []
    for i, r in enumerate(records, start=1):
        for field in ["image_path", "ground_truth", "category"]:
            if field not in r or str(r[field]).strip() == "":
                raise ValueError(f"Ligne {i}: champ manquant {field}")
        src = assert_inside(base_dir / str(r["image_path"]), base_dir)
        if not src.is_file():
            matches = list(base_dir.rglob(Path(str(r["image_path"])).name))
            if len(matches) == 1: src = matches[0]
            else: raise FileNotFoundError(f"Image introuvable: {r['image_path']}")
        if src.suffix.lower() not in ALLOWED_EXTENSIONS:
            raise ValueError(f"Format non supporté: {src}")
        dest = UPLOAD_DIR / f"{len(rows):05d}_{src.name}"
        shutil.copy2(src, dest)
        rows.append({
            "image_path": str(dest),
            "ground_truth": str(r["ground_truth"]),
            "category": str(r["category"]).strip().lower().replace(" ", "_"),
            "description": str(r.get("description", "Dataset importé")),
        })
    return rows


def import_dataset_zip():
    from google.colab import files
    print("Uploadez votre ZIP dataset.")
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
    if not zip_name: raise ValueError("Aucun ZIP reçu.")
    extract_dir = WORK_DIR / "custom_dataset_upload"
    if extract_dir.exists(): shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    safe_extract_zip(zip_name, extract_dir)
    return read_records_from_folder(extract_dir)


def download_kaggle_dataset(handle: str):
    import kagglehub
    if not handle: raise ValueError("KAGGLE_DATASET_HANDLE est vide.")
    path = Path(kagglehub.dataset_download(handle))
    print("Dataset Kaggle téléchargé:", path)
    return read_records_from_folder(path)

print("Fonctions dataset prêtes.")

In [ ]:
# ÉTAPE 3C — Construire le dataset actif
records = []
if USE_SAMPLE_DATASET:
    records.extend(build_sample_dataset())
if IMPORT_CUSTOM_DATASET_ZIP:
    records.extend(import_dataset_zip())
if DOWNLOAD_KAGGLE_DATASET:
    records.extend(download_kaggle_dataset(KAGGLE_DATASET_HANDLE))
if not records:
    raise ValueError("Aucun dataset chargé.")

dataset_df = pd.DataFrame(records)
display(dataset_df[["image_path", "category", "description"]])
display(dataset_df["category"].value_counts().rename("documents").to_frame())

## ÉTAPE 4 — Modèles demandés

Cellules prêtes pour télécharger/préparer :

- Qwen OCR 0.8B : `loay/English-Document-OCR-Qwen3.5-0.8B`. Remarque : `Qwen/Qwen3.5-0.8B` seul est texte-only, donc pas OCR image direct.
- MiniCPM-V 4.6 : `openbmb/MiniCPM-V-4.6`.
- Chandra OCR 2 : `datalab-to/chandra-ocr-2`.
- LightOnOCR : `lightonai/LightOnOCR-2-1B`.
- dots.ocr / DOT OCR : `rednote-hilab/dots.ocr`.
- PaddleOCR classique : paquet `paddleocr`.
- PaddleOCR-VL : `PaddlePaddle/PaddleOCR-VL-1.6`.
- LocateAnything : `nvidia/LocateAnything-3B`, plutôt localisation/grounding que OCR pur.
- Unlimited OCR : `baidu/Unlimited-OCR`.

Par défaut, seuls `mock` et `easyocr` sont activés pour éviter d’exploser la RAM Colab. Active les autres un par un.

In [ ]:
# ÉTAPE 4A — Catalogue modèles
MODEL_CATALOG = {
    "mock": {"enabled": True, "kind": "mock", "model_id": "mock-local", "note": "Test pipeline."},
    "easyocr": {"enabled": True, "kind": "easyocr", "model_id": "easyocr-fr-en", "note": "Baseline OCR CPU/GPU."},
    "paddleocr_classic": {"enabled": False, "kind": "paddleocr", "model_id": "paddleocr-classic", "note": "Nécessite INSTALL_PADDLE=True."},
    "qwen_ocr_0_8b": {"enabled": False, "kind": "hf_pipeline", "model_id": "loay/English-Document-OCR-Qwen3.5-0.8B", "note": "OCR basé Qwen3.5 0.8B."},
    "minicpm_v_4_6": {"enabled": False, "kind": "hf_pipeline", "model_id": "openbmb/MiniCPM-V-4.6", "note": "VLM; peut demander beaucoup de VRAM."},
    "chandra_ocr_2": {"enabled": False, "kind": "hf_pipeline", "model_id": "datalab-to/chandra-ocr-2", "note": "OCR/document parsing."},
    "lightonocr_2_1b": {"enabled": False, "kind": "hf_pipeline", "model_id": "lightonai/LightOnOCR-2-1B", "note": "OCR LightOn."},
    "dots_ocr": {"enabled": False, "kind": "hf_download_only", "model_id": "rednote-hilab/dots.ocr", "note": "Peut demander code spécifique."},
    "paddleocr_vl": {"enabled": False, "kind": "hf_download_only", "model_id": "PaddlePaddle/PaddleOCR-VL-1.6", "note": "PaddleOCR-VL, intégration spécifique."},
    "locateanything_3b": {"enabled": False, "kind": "hf_download_only", "model_id": "nvidia/LocateAnything-3B", "note": "Localisation, pas OCR pur."},
    "unlimited_ocr": {"enabled": False, "kind": "hf_pipeline", "model_id": "baidu/Unlimited-OCR", "note": "OCR HF."},
}

for name, cfg in MODEL_CATALOG.items():
    print(f"{name:20s} enabled={cfg['enabled']} kind={cfg['kind']} id={cfg['model_id']}")

In [ ]:
# ÉTAPE 4B — Télécharger les poids des modèles Hugging Face activés
from huggingface_hub import snapshot_download

DOWNLOAD_MODEL_WEIGHTS = True
HF_LOCAL_DIR = WORK_DIR / "hf_models"
HF_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

downloaded_models = {}
if DOWNLOAD_MODEL_WEIGHTS:
    for name, cfg in MODEL_CATALOG.items():
        if not cfg["enabled"]: continue
        if not cfg["kind"].startswith("hf_"):
            print(f"{name}: pas de snapshot HF nécessaire."); continue
        print(f"Téléchargement: {name} -> {cfg['model_id']}")
        try:
            downloaded_models[name] = snapshot_download(repo_id=cfg["model_id"], local_dir=str(HF_LOCAL_DIR / name), resume_download=True)
            print("OK:", downloaded_models[name])
        except Exception as exc:
            print("Échec:", exc)
else:
    print("Téléchargement désactivé.")

In [ ]:
# ÉTAPE 5 — Métriques
try:
    import Levenshtein
except Exception:
    Levenshtein = None

def normalize_text(text):
    return " ".join(str(text or "").strip().split()).lower()

def edit_distance(a, b):
    if Levenshtein:
        return Levenshtein.distance(a, b)
    dp = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = min(dp[j]+1, dp[j-1]+1, prev + (ca != cb))
            prev = cur
    return dp[-1]

def cer(expected, predicted):
    e = normalize_text(expected); p = normalize_text(predicted)
    return 0.0 if not e and not p else (1.0 if not e else edit_distance(e, p)/len(e))

def wer(expected, predicted):
    e = normalize_text(expected).split(); p = normalize_text(predicted).split()
    return 0.0 if not e and not p else (1.0 if not e else edit_distance("\n".join(e), "\n".join(p))/len(e))

def exact_match(expected, predicted):
    return int(normalize_text(expected) == normalize_text(predicted))

print("Métriques prêtes: CER, WER, exact match, latency.")

In [ ]:
# ÉTAPE 6 — Adaptateurs modèles
OCR_PROMPT = "You are a professional OCR engine. Transcribe all visible text. Return only the transcription. Preserve line breaks and tables when possible."

class BaseAdapter:
    def __init__(self, name, cfg): self.name=name; self.cfg=cfg
    def predict(self, image_path): raise NotImplementedError

class MockAdapter(BaseAdapter):
    def predict(self, image_path):
        start=time.time()
        return {"text": Path(image_path).stem, "latency": time.time()-start, "status":"success", "raw_response":"mock", "error":None, "device":DEVICE}

class EasyOCRAdapter(BaseAdapter):
    def __init__(self, name, cfg):
        super().__init__(name,cfg)
        import easyocr
        self.reader = easyocr.Reader(["fr","en"], gpu=(DEVICE=="cuda"))
    def predict(self, image_path):
        start=time.time()
        try:
            out = self.reader.readtext(image_path, detail=0)
            return {"text":"\n".join(out), "latency":time.time()-start, "status":"success", "raw_response":repr(out), "error":None, "device":DEVICE}
        except Exception as exc:
            return {"text":"", "latency":time.time()-start, "status":"failed", "raw_response":None, "error":str(exc), "device":DEVICE}

class PaddleOCRAdapter(BaseAdapter):
    def __init__(self, name, cfg):
        super().__init__(name,cfg)
        from paddleocr import PaddleOCR
        self.ocr = PaddleOCR(use_angle_cls=True, lang="fr")
    def predict(self, image_path):
        start=time.time()
        try:
            result = self.ocr.ocr(image_path, cls=True)
            lines=[]
            for page in result or []:
                for item in page or []:
                    if len(item) >= 2: lines.append(str(item[1][0]))
            return {"text":"\n".join(lines), "latency":time.time()-start, "status":"success", "raw_response":repr(result), "error":None, "device":DEVICE}
        except Exception as exc:
            return {"text":"", "latency":time.time()-start, "status":"failed", "raw_response":None, "error":str(exc), "device":DEVICE}

class HFPipelineAdapter(BaseAdapter):
    def __init__(self, name, cfg):
        super().__init__(name,cfg)
        from transformers import pipeline
        kwargs = {"trust_remote_code": True}
        if torch and DEVICE == "cuda": kwargs["device"] = 0
        self.pipe = pipeline("image-text-to-text", model=cfg["model_id"], **kwargs)
    def predict(self, image_path):
        start=time.time()
        try:
            img = Image.open(image_path).convert("RGB")
            response = self.pipe({"images": img, "text": OCR_PROMPT}, max_new_tokens=1024)
            text = response
            if isinstance(response, list) and response:
                x=response[0]; text=x.get("generated_text", x.get("text", str(x))) if isinstance(x,dict) else str(x)
            elif isinstance(response, dict):
                text=response.get("generated_text", response.get("text", str(response)))
            return {"text":str(text).strip(), "latency":time.time()-start, "status":"success", "raw_response":repr(response), "error":None, "device":DEVICE}
        except Exception as exc:
            return {"text":"", "latency":time.time()-start, "status":"failed", "raw_response":None, "error":str(exc), "device":DEVICE}

class DownloadOnlyAdapter(BaseAdapter):
    def predict(self, image_path):
        return {"text":"", "latency":0.0, "status":"skipped", "raw_response":None, "error":"Poids téléchargés, mais adaptateur spécifique requis.", "device":DEVICE}

def build_adapter(name,cfg):
    if cfg["kind"] == "mock": return MockAdapter(name,cfg)
    if cfg["kind"] == "easyocr": return EasyOCRAdapter(name,cfg)
    if cfg["kind"] == "paddleocr": return PaddleOCRAdapter(name,cfg)
    if cfg["kind"] == "hf_pipeline": return HFPipelineAdapter(name,cfg)
    if cfg["kind"] == "hf_download_only": return DownloadOnlyAdapter(name,cfg)
    raise ValueError(cfg["kind"])

print("Adaptateurs prêts.")

In [ ]:
# ÉTAPE 7 — Sélection documents
SELECTION_MODE = "Quantité globale"  # Tout le dataset | Quantité globale | Par catégorie
GLOBAL_QUANTITY = 3
CATEGORY_QUANTITIES = {"bank": 2, "tables": 2, "handwritten_form": 2}
SHUFFLE = True
SEED = 42

def select_dataset(df):
    if SELECTION_MODE == "Tout le dataset": return df.reset_index(drop=True)
    if SELECTION_MODE == "Quantité globale":
        x = df.sample(frac=1, random_state=SEED) if SHUFFLE else df
        return x.head(min(GLOBAL_QUANTITY, len(x))).reset_index(drop=True)
    if SELECTION_MODE == "Par catégorie":
        parts=[]
        for cat,n in CATEGORY_QUANTITIES.items():
            part=df[df.category==cat]
            if SHUFFLE: part=part.sample(frac=1, random_state=SEED)
            parts.append(part.head(int(n)))
        return pd.concat(parts, ignore_index=True) if parts else df.iloc[0:0]
    raise ValueError(SELECTION_MODE)

selected_df = select_dataset(dataset_df)
print(f"Documents sélectionnés: {len(selected_df)} / {len(dataset_df)}")
display(selected_df[["image_path","category","description"]])

In [ ]:
# ÉTAPE 8 — Lancer le benchmark
RUN_ID = time.strftime("%Y%m%d-%H%M%S")
run_dir = RUNS_DIR / RUN_ID
run_dir.mkdir(parents=True, exist_ok=True)
active_models = {n:c for n,c in MODEL_CATALOG.items() if c["enabled"]}
print("Modèles actifs:", list(active_models))

results=[]; adapters={}
for name,cfg in active_models.items():
    print(f"Chargement: {name}")
    try:
        adapters[name]=build_adapter(name,cfg)
    except Exception as exc:
        print("Échec chargement", name, exc)
        for _,row in selected_df.iterrows():
            results.append({"run_id":RUN_ID,"model":name,"model_id":cfg["model_id"],"image_path":row.image_path,"category":row.category,"ground_truth":row.ground_truth,"text":"","latency":0.0,"status":"failed_load","raw_response":None,"error":str(exc),"device":DEVICE})

for name,adapter in adapters.items():
    cfg=active_models[name]
    for i,row in selected_df.iterrows():
        print(f"[{name}] {i+1}/{len(selected_df)} {Path(row.image_path).name}")
        pred=adapter.predict(row.image_path)
        r={"run_id":RUN_ID,"model":name,"model_id":cfg["model_id"],"image_path":row.image_path,"category":row.category,"ground_truth":row.ground_truth,**pred}
        r["cer"]=cer(r["ground_truth"], r["text"]); r["wer"]=wer(r["ground_truth"], r["text"]); r["exact_match"]=exact_match(r["ground_truth"], r["text"])
        results.append(r)
        pd.DataFrame(results).to_json(run_dir/"results.json", orient="records", force_ascii=False, indent=2)

results_df=pd.DataFrame(results)
for col, default in [("cer",1.0),("wer",1.0),("exact_match",0)]:
    if col not in results_df: results_df[col]=default
results_df.to_csv(run_dir/"details.csv", index=False)
display(results_df[["model","category","status","latency","cer","wer","exact_match","error"]])
print("Run dir:", run_dir)

In [ ]:
# ÉTAPE 9 — Résumé par modèle
summary_df = (results_df.groupby("model").agg(
    evaluations=("model","count"),
    success_rate=("status", lambda s: float((s=="success").mean())),
    avg_latency=("latency","mean"),
    p95_latency=("latency", lambda s: float(np.percentile(s,95)) if len(s) else np.nan),
    avg_cer=("cer","mean"),
    avg_wer=("wer","mean"),
    exact_match_rate=("exact_match","mean"),
).reset_index().sort_values(["success_rate","avg_cer","avg_latency"], ascending=[False,True,True]))
summary_df.to_csv(run_dir/"summary.csv", index=False)
display(summary_df)
if len(summary_df):
    best=summary_df.iloc[0]
    print(f"Recommandation simple: {best['model']} | CER={best['avg_cer']:.3f} | latence={best['avg_latency']:.2f}s | réussite={best['success_rate']:.0%}")

In [ ]:
# ÉTAPE 10 — Graphiques
import plotly.express as px
px.scatter(summary_df, x="avg_latency", y="avg_cer", size="evaluations", color="model", hover_data=["success_rate","avg_wer"], title="Qualité vs vitesse — plus bas/gauche = meilleur").show()
px.bar(summary_df, x="model", y="success_rate", title="Taux de réussite technique").show()
px.box(results_df, x="model", y="latency", color="model", title="Distribution du temps par image").show()

In [ ]:
# ÉTAPE 11 — Explorer un résultat
from IPython.display import display, Markdown
RESULT_INDEX = 0
row = results_df.iloc[RESULT_INDEX]
print("Modèle:", row.model)
print("Image:", row.image_path)
print("CER:", row.cer, "WER:", row.wer, "Latence:", row.latency)
display(Image.open(row.image_path))
display(Markdown("### Texte attendu\n```text\n" + str(row.ground_truth) + "\n```"))
display(Markdown("### Texte extrait\n```text\n" + str(row.text) + "\n```"))
if row.error: print("Erreur:", row.error)

In [ ]:
# ÉTAPE 12 — Télécharger les résultats
from google.colab import files
archive = shutil.make_archive(str(run_dir), "zip", run_dir)
files.download(archive)
print("Archive:", archive)

In [ ]:
# ÉTAPE 13 — Interface Gradio minimale
import gradio as gr
choices = [f"{i}: {Path(r.image_path).name} | {r.model} | CER={r.cer:.2f}" for i,r in results_df.reset_index(drop=True).iterrows()]

def view_result(choice):
    idx=int(str(choice).split(":",1)[0]); r=results_df.iloc[idx]
    metrics=f"Modèle: {r['model']}\nStatus: {r['status']}\nLatence: {r['latency']:.2f}s\nCER: {r['cer']:.3f}\nWER: {r['wer']:.3f}\nExact match: {r['exact_match']}"
    return r.image_path, metrics, r.ground_truth, r.text, str(r.raw_response)

with gr.Blocks(title="OCR Benchmark Standalone") as demo:
    gr.Markdown("# OCR Benchmark — résultats")
    selector=gr.Dropdown(choices, value=choices[0] if choices else None, label="Résultat")
    with gr.Row():
        img=gr.Image(label="Image", type="filepath")
        metrics=gr.Textbox(label="Métriques", lines=8)
    with gr.Row():
        expected=gr.Textbox(label="Texte attendu", lines=12)
        extracted=gr.Textbox(label="Texte extrait", lines=12)
    raw=gr.Textbox(label="Sortie brute", lines=8)
    selector.change(view_result, selector, [img,metrics,expected,extracted,raw])
    if choices:
        demo.load(lambda: view_result(choices[0]), None, [img,metrics,expected,extracted,raw])

demo.launch(share=True, debug=False)